In [1]:
import sys, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:/", repo_root)
from fixedincomelib import *

print("Fixed Income Library is loaded.")

Added to sys.path:/ c:\QuantBricker
Fixed Income Library is loaded.


### Create Build Method Collection

In [2]:
bm_list = []
# create yield curve build method
content_sofr = {
    "TARGET": "SOFR-1B",
    "OVERNIGHT INDEX FUTURE": "SOFR-FUTURE-3M",
    "OVERNIGHT INDEX SWAP": "USD-SOFR-OIS",
}
bm_list.append(qfCreateBuildMethod("YIELD_CURVE_INDEX", content_sofr))
# funding build method
content_sofr_funding = {
    "TARGET": "SOFR-1B-FLAT",
    "SPREAD ZERO RATE": "SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD",
}
bm_list.append(qfCreateBuildMethod("YIELD_CURVE_FUNDING", content_sofr_funding))
# create yield curve common build method
content = {"TARGET": "USD", "FUNDING PARAMETERS": "USD-FUNDING-PARAMETERS", "SOLVER": "BRENTQ"}
bm_list.append(qfCreateBuildMethod("YIELD_CURVE_COMMON", content))
build_method_collection = qfCreateModelBuildMethodCollection(bm_list)

### Create Data Collection

In [3]:
### ois futures
df_fut = pd.DataFrame(
    [
        ["2026-03-19x2026-06-18", 96.44],
        ["2026-06-18x2026-09-17", 96.70],
        ["2026-09-17x2026-12-10", 96.85],
        ["2026-12-10x2027-03-17", 96.90],
        ["2027-03-17x2027-06-16", 96.91],
        ["2027-06-16x2027-09-15", 96.89],
        ["2027-09-15x2027-12-15", 96.85],
        ["2027-12-15x2028-03-15", 96.81],
        ["2028-03-15x2028-06-21", 96.76],
        ["2028-06-21x2028-09-20", 96.71],
        ["2028-09-20x2028-12-20", 96.66],
        ["2028-12-20x2029-03-21", 96.61],
    ],
    columns=["axis1", "values"],
).set_index("axis1")
data_fut = qfCreateData1D("OVERNIGHT INDEX FUTURE", "SOFR-FUTURE-3M", df_fut)

In [4]:
### ois swap
df_swap = pd.DataFrame(
    [
        ["4Y", 0.03358],
        ["5Y", 0.03422],
        ["6Y", 0.03491],
        ["7Y", 0.03560],
        ["8Y", 0.03624],
        ["9Y", 0.03685],
        ["10Y", 0.03742],
        ["12Y", 0.03849],
        ["15Y", 0.03974],
        ["20Y", 0.04087],
        ["25Y", 0.04110],
        ["30Y", 0.04089],
        ["35Y", 0.04044],
        ["40Y", 0.03996],
        ["50Y", 0.03887],
        ["60Y", 0.03772],
    ],
    columns=["axis1", "values"],
).set_index("axis1")
data_swap = qfCreateData1D("OVERNIGHT INDEX SWAP", "USD-SOFR-OIS", df_swap)

In [5]:
# spread zero rate
df_spread_zero_rate_rfr = pd.DataFrame(
    [["1Y", 0.0], ["5Y", 0.0], ["10Y", 0.0], ["20Y", 0.0], ["30Y", 0.0]],
    columns=["axis1", "values"],
).set_index("axis1")
data_szr_rfr = qfCreateData1D(
    "SPREAD ZERO RATE", "SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD", df_spread_zero_rate_rfr
)

In [6]:
### funding parameter table
df_dg = pd.DataFrame(
    [
        ["Overnight Index Future", "SOFR-FUTURE-3M", "SOFR-1B-FLAT"],
        ["Overnight Index Swap", "USD-SOFR-OIS", "SOFR-1B-FLAT"],
    ],
    columns=["DATA TYPE", "DATA CONVENTION", "FUNDING IDENTIFIER"],
)
data_fpt = qfCreateDataGeneric("DATA GENERIC", "USD-FUNDING-PARAMETERS", df_dg)

In [7]:
### pack up all data into data collection
data_collection = qfCreateDataCollection([data_szr_rfr, data_fut, data_swap, data_fpt])

### Create Model Yield Curve

In [8]:
value_date = "2026-02-11"
yc_model = qfCreateModel(value_date, "YIELD_CURVE", data_collection, build_method_collection)
path = "serialized/yc_model_calibrated.pickle"
qfWriteModelObjectToFile(yc_model, path)

'DONE'

In [20]:
yc_model.calculate_model_jacobian()

In [24]:
pd.DataFrame(yc_model.model_jacobian).to_clipboard()

### Re-Price Calibration Instruments

In [9]:
### everything is collateralised in RFR
fi_vp = qfCreateValuationParameters("FUNDING INDEX PARAMETER", {"Funding Index": "SOFR-1B-FLAT"})
vp_collection = qfCreateValuationParametersCollection([fi_vp])
### re-price each product
req = "parrateorspread"
dc_display = data_collection.display()
for data in data_collection:
    if data.data_shape != "DATAGENERIC" and data.data_type != "SPREAD ZERO RATE":
        for _, row in data.display().iterrows():
            axis, quote = row["axis1"], row["values"]
            this_product = qfCreateProductFromDataConvention(
                value_date, data.data_convention.name, axis, quote
            )
            par_rate = qfCreateValueReport(yc_model, this_product, vp_collection, req)
            if "FUTURE" in data.data_type:
                print(f"target is {quote:.2f}, and model implied is {100 - 100 * par_rate:.2f}.")
            else:
                print(f"target is {quote:.2%}, and model implied is {par_rate:.2%}.")
                pv = qfCreateValueReport(yc_model, this_product, vp_collection, "pv")
                print(f"Pv of the swap is {pv[0][1]:.2f}.")

target is 96.44, and model implied is 96.44.
target is 96.70, and model implied is 96.70.
target is 96.85, and model implied is 96.85.
target is 96.90, and model implied is 96.90.
target is 96.91, and model implied is 96.91.
target is 96.89, and model implied is 96.89.
target is 96.85, and model implied is 96.85.
target is 96.81, and model implied is 96.81.
target is 96.76, and model implied is 96.76.
target is 96.71, and model implied is 96.71.
target is 96.66, and model implied is 96.66.
target is 96.61, and model implied is 96.61.
target is 3.36%, and model implied is 3.36%.
Pv of the swap is -0.00.
target is 3.42%, and model implied is 3.42%.
Pv of the swap is -0.00.
target is 3.49%, and model implied is 3.49%.
Pv of the swap is -0.00.
target is 3.56%, and model implied is 3.56%.
Pv of the swap is -0.00.
target is 3.62%, and model implied is 3.62%.
Pv of the swap is -0.00.
target is 3.69%, and model implied is 3.69%.
Pv of the swap is -0.00.
target is 3.74%, and model implied is 3.

# Fixed Accrued test

In [10]:
# parametric product
effecitve_date = "2026-03-26"
termination_date = "2026-05-26"
currency = "USD"
notional = 1e6
accrual_basis = "ACT/360"
business_day_convention = "F"
holiday_convention = "USGS"
pay_offset = "2D"
payment_date = qfAddPeriod(
    termination_date, pay_offset, business_day_convention, holiday_convention
)
prod_fixed_accrued = qfCreateProducFixedAccrued(
    effecitve_date, termination_date, currency, notional, accrual_basis
)

In [11]:
display(qfDisplayProduct(prod_fixed_accrued))

,Name,Value
0,Product Type,PRODUCT_FIXED_ACCRUED
1,Notional,1000000.0
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-03-26
5,Termination Date,2026-05-26
6,Accrual Basis,ACT/360
7,Payment Date,2026-05-26
8,Business Day Convention,F
9,Holiday Convention,USGS


In [12]:
qfCreateValueReport(yc_model, prod_fixed_accrued, vp_collection, "pv")

[['USD', 167718.44750591737]]

In [13]:
### test risk
risk = qfCreateValueReport(yc_model, prod_fixed_accrued, vp_collection, "firstorderrisk")
df_risk = risk.display()
df_risk.VALUES = df_risk.VALUES.round(10)
display(df_risk)

,DATA_TYPE,DATA_CONVENTION,AXIS1,AXIS2,MARKET_QUOTE,UNIT,VALUES
0,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,1Y,,0.0,0.0001,-4.778827
1,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,5Y,,0.0,0.0001,0.000000
2,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,10Y,,0.0,0.0001,0.000000
3,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,20Y,,0.0,0.0001,0.000000
4,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,30Y,,0.0,0.0001,0.000000
5,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-03-19x2026-06-18,,96.44,-0.01,-4.801987
6,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-06-18x2026-09-17,,96.7,-0.01,-0.000000
7,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-09-17x2026-12-10,,96.85,-0.01,-0.000000
8,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-12-10x2027-03-17,,96.9,-0.01,-0.000000
9,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2027-03-17x2027-06-16,,96.91,-0.01,-0.000000


In [14]:
qfCreateValueReport(yc_model, prod_fixed_accrued, vp_collection, "cashflowsreport").display().T

,0
PRODUCT_TYPE,PRODUCT_FIXED_ACCRUED
VALUATION_ENGINE_TYPE,ValuationEngineProductFixedAccrued
LEG_ID,0
CASHFLOW_ID,0
PAY_OR_RECEIVE,1.0
NOTIONAL,1000000.0
PAY_DATE,"May 26th, 2026"
FORECASTED_AMOUNT,169444.444444
PV,167718.447506
DISCOUNG FACTOR,0.989814


In [15]:
engine = ValuationEngineProductRegistry().get(
    ("YIELD_CURVE", "PRODUCT_FIXED_ACCRUED", "ANALYTIC PARAMETER")
)(
    model=yc_model,
    product=prod_fixed_accrued,
    valuation_parameters_collection=vp_collection,
    request=req,
)

In [16]:
engine.calculate_value()

In [17]:
pv_cash = engine.get_value_and_cash()
cf_report = engine.create_cash_flows_report()

In [18]:
cf_report.display().T

,0
PRODUCT_TYPE,PRODUCT_FIXED_ACCRUED
VALUATION_ENGINE_TYPE,ValuationEngineProductFixedAccrued
LEG_ID,0
CASHFLOW_ID,0
PAY_OR_RECEIVE,1.0
NOTIONAL,1000000.0
PAY_DATE,"May 26th, 2026"
FORECASTED_AMOUNT,169444.444444
PV,167718.447506
DISCOUNG FACTOR,0.989814


# Bond 

In [19]:
bond = qfCreateProductBond(
    isin="US9128284W34",
    settlement_date=Date("2026-02-24"),
    buy_sell="LONG",
)
bond.currency

TypeError: qfCreateProductBond() got an unexpected keyword argument 'isin'

In [ ]:
qfCreateValueReport(yc_model, bond, vp_collection, "pvdetailed").display()

,Currency,Type,Value
0,USD,PV,100.482855
1,USD,CASH,0.000000
